# Sarsa

## Derive TD Target


折扣回报：
$$
\begin{align}
U_t = R_t + \gamma R_{t+1} + \gamma ^ 2 R_{t+2} + \gamma ^ 3 R_{t+3} + \gamma ^ 4 R_{t+4}... \\
= R_t + \gamma (R_{t+1} + \gamma R_{t+2} + \gamma ^ 2 R_{t+3} + \gamma ^ 3 R_{t+4}...) \\
= R_t + \gamma U_{t+1}
\end{align}
$$

- 假设$R_t$依赖$(S_t, A_t, S_{t+1})$；
- $Q_\pi(s_t, a_t) = \mathbb E [U_t| s_t, a_t] = \mathbb E [R_t + \gamma \cdot U_{t+1}| s_t, a_t] = \mathbb E [R_t | s_t, a_t] + \gamma \cdot \mathbb E [U_{t+1}| s_t, a_t] = \mathbb E [R_t | s_t, a_t] + \gamma \cdot \mathbb E[Q_\pi(S_{t+1}, A_{t+1}| s_t, a_t)]$

得到等式：
$$
Q_\pi(s_t, a_t) = \mathbb E [R_t + \gamma \cdot Q_\pi(S_{t+1}, A_{t+1})]
$$
对任何$\pi$均成立。

等式左边$Q_\pi(s_t, a_t)$为$t$时刻的动作价值，可以等价写为右边的形式，等式两边都存在$Q_\pi$形式。

等式右边为一个期望，此期望是针对$S_{t+1}$和$A_{t+1}$求出，直接求期望十分困难，故采取对期望求蒙特卡洛近似。

- 取$R_t \approx r_t$
- 取$Q_\pi(S_{t+1}, A_{t+1}) \approx Q_\pi(s_{t+1}, a_{t+1})$
- `TD target`：$y_t = r_t + Q_\pi(s_{t+1}, a_{t+1}) \approx \mathbb E [R_t + \gamma \cdot Q_\pi(S_{t+1}, A_{t+1})]$

$Q_\pi(s_t, a_t)$完全为估计，而$y_t$部分基于真实奖励，故可认为$y_t$更为可靠。固定$y_t$作为观测到的常数值，改变动作价值$Q_\pi(s_t, a_t)$使其更接近$y_t$，这就是`TD`算法的思路。

## Sarsa: Tabular Version

- 想要学习$Q_\pi(s, a)$
- 假设状态和动作的数量都是有限的，可以画出一个表格：

|             | Action $a_1$ | Action $a_2$ | Action $a_3$ | Action $a_4$ | ... |
|-------------|--------------|--------------|--------------|--------------|-----|
| State $s_1$ |              |              |              |              |     |
| State $s_2$ |              |              |              |              |     |
| State $s_3$ |              |              |              |              |     |
| ...         |              |              |              |              |     |

表中的一行对应一个状态，一列对应一个动作，表中每个元素表示一个动作价值，即$Q_\pi(s_t, a_t)$。我们要做的即通过`Sarsa`算法更新表格，每次更新一个元素。
- 每次观测到一个四元组`transition`：$(s_t, a_t, r_t, s_{t+1})$；
- 通过策略函数$\pi$进行抽样采取动作：$a_{t+1} \sim \pi(\cdot | s_{t+1})$；
- 计算`TD target`：$y_t = r_t + \gamma \cdot Q_\pi(s_{t+1j, a_{t+1}})$；
- 计算`TD error`：$\delta_t = Q_\pi(s_t, a_t) - y_t$；
- 更新：$Q_\pi(s_t, a_t) \leftarrow Q_\pi(s_t, a_t) - \alpha \cdot \delta_t$.

每次通过$(s_t, a_t, r_t, s_{s+1}, a_{t+1})$更新$Q_\pi$，即**State-Action-Reward-State-Action(SARSA)**.

## Sarsa: Neural Network Version

- 使用价值网络$q(s, a; w)$拟合行为价值函数$Q_\pi(s, a)$，输入为当前状态，输出为所有行为的价值；
- `TD target`：$y_t = r_t + \gamma \cdot q(s_{t+1}, a+{t+1}; w)$；
- `TD error`：$\delta_t = q(s_t, a_t; w) - y_t$；
- $Loss = \frac {\delta_t^2} 2$；
- 梯度：$\frac {\partial \delta_t^2 / 2}{\partial w} = \delta_t \cdot \frac {\partial q(s_t, a_t; w)}{\partial w}$；
- 梯度下降：$w \leftarrow w - \alpha \cdot \delta_t \cdot \frac {\partial q(s_t, a_t; w)}{\partial w}$.